### This is the multi agent system building 
1. Policy agent
2. Logistic agent
3. Booking tool

Include router and llm judge also

### 1. Import essential libraries and configs 

In [1]:
import os
import sys

from typing import List, Dict, Any
from pathlib import Path

sys.path.append("..")

from utils import get_api_key, get_model_configuration, get_parameter_configuration, LLM
from tools.web_search import WebSearchTool , get_web_search_tool
from tools.booking import get_booking_tool
from rag.retriever import MedicalKnowladgeRetriever

ROOT_DIR = Path.cwd() if (Path.cwd() / "data" / "medical_guides").exists() else Path.cwd().parent
DOCS_DIR = ROOT_DIR / "data" / "medical_guides"

In [3]:
llm = LLM()
web_search = WebSearchTool()
booking_tool = get_booking_tool()
retriever = MedicalKnowladgeRetriever(docs_dir=str(DOCS_DIR))

### 2. Implement Policy Agent

In [ ]:
class PolicyAgent:
    """
    This is for Internal knowladge: RAG  
    """
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
        self.name = "Policy Agent"

    def process(self, query):
        """
        query : This is user's message
        based on the user's query find the proper answer from RAG internal
        """
        context = self.retriever.get_context(query, max_k=3)

        system_prompt = """
        You are a medical policy specialist,
        Use internal knowladge for answering to the questions about procedure, guidelines, and clinical protocols.
        Be specific and cite sources. 
        """.strip()

        prompt = f"""
        query : {query}
        Internal knowladge : {context}
        Provide a focused answer about medical policies and procedures.
        """.strip()

        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)

        return {
            "agent": self.name,
            "response": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"],
            "source_type": "internal_rag"
        }

        

### 3. Implement Logistic Agent

In [ ]:
class LogisticAgent:
    """
    Specialist for operational logistics using web search.
    """
    def __init__(self, llm, web_search):
        self.llm = llm
        self.web_search = web_search
        self.name = "Logistic Agent"

    def process(self, query):
        """
        Provide answer to user's query from searching web 
        """
        